In [7]:
#visuallize by time when people were in the area and what type of places were opne at that time devided between weekdays and weekends and public holidays and non public holidays
import pandas as pd
import plotly.express as px
from pathlib import Path

# ----------------------------
# 1. Load data
# ----------------------------

movement = pd.read_csv("movement_stats_hourly.csv")
locations = pd.read_csv("Location_data/akseltorv_locations_cleaned_data.csv")
holidays = pd.read_csv("Holiday_data/Holidays_data_updated")

# ----------------------------
# 2. Prepare movement data
# ----------------------------

movement["timestamp"] = pd.to_datetime(
    movement["timestamp"],
    utc=True
).dt.tz_convert("Europe/Copenhagen")

movement["date"] = movement["timestamp"].dt.date
movement["hour"] = movement["timestamp"].dt.hour
movement["weekday_num"] = movement["timestamp"].dt.weekday
movement["weekday_short"] = movement["timestamp"].dt.day_name().str.lower().str[:3]

# Prepare holidays
holidays["date"] = pd.to_datetime(holidays["date"]).dt.date
holiday_dates = set(holidays["date"])

movement["is_public_holiday"] = movement["date"].isin(holiday_dates)

# Divide into weekday, weekend, and public holiday
movement["day_group"] = "Weekday"
movement.loc[movement["weekday_num"] >= 5, "day_group"] = "Weekend"
movement.loc[movement["is_public_holiday"], "day_group"] = "Public holiday"

# Use pedestrians as the measure of "people in the city"
pedestrian_hourly = (
    movement[movement["category"] == "pedestrian"]
    .groupby(["day_group", "weekday_short", "hour"], as_index=False)["amount"]
    .mean()
    .rename(columns={"amount": "avg_pedestrian_movement"})
)

# ----------------------------
# 3. Clean place type names
# ----------------------------

type_labels = {
    "food": "Restaurants / cafés",
    "retail": "Retail",
    "service": "Services",
    "public service": "Public services",
    "school": "Schools",
    "parking": "Parking",
    "nightclub": "Nightlife"
}

locations["place_type"] = locations["type"].map(type_labels).fillna(locations["type"])

# ----------------------------
# 4. Helper functions for opening hours
# ----------------------------

def time_to_hour(value):
    """
    Converts time values like '10:00' into hour numbers like 10.
    Treats missing values and 'closed' as None.
    """
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    if value in ["closed", ""]:
        return None

    try:
        return int(value.split(":")[0])
    except Exception:
        return None


def is_open_at_hour(row, day, hour, is_public_holiday=False):
    """
    Checks if a place is open at a specific hour.

    Example:
    If a shop is open from 10:00 to 18:00,
    then it is counted as open from 10 to 17.
    """

    if is_public_holiday:
        public_open = row.get("public_holiday_open")

        if pd.isna(public_open) or str(public_open).strip().lower() in ["closed", ""]:
            return False

        public_close = row.get("public_holiday_close")

        if public_close is not None and not pd.isna(public_close):
            open_hour = time_to_hour(public_open)
            close_hour = time_to_hour(public_close)
        else:
            # If no exact public holiday closing time exists,
            # use Sunday opening hours as a fallback
            open_hour = time_to_hour(row.get("open_sun"))
            close_hour = time_to_hour(row.get("close_sun"))

    else:
        open_hour = time_to_hour(row.get(f"open_{day}"))
        close_hour = time_to_hour(row.get(f"close_{day}"))

    if open_hour is None or close_hour is None:
        return False

    # Normal opening hours, for example 10:00–18:00
    if open_hour < close_hour:
        return open_hour <= hour < close_hour

    # Overnight opening hours, for example 22:00–04:00
    if open_hour > close_hour:
        return hour >= open_hour or hour < close_hour

    return False

# ----------------------------
# 5. Count open place types by hour
# ----------------------------

rows = []

day_group_weekdays = {
    "Weekday": ["mon", "tue", "wed", "thu", "fri"],
    "Weekend": ["sat", "sun"],
    "Public holiday": ["sun"]
}

place_types = sorted(locations["place_type"].dropna().unique())

for day_group, days in day_group_weekdays.items():
    for day in days:
        for hour in range(24):

            open_mask = locations.apply(
                lambda row: is_open_at_hour(
                    row,
                    day,
                    hour,
                    is_public_holiday=(day_group == "Public holiday")
                ),
                axis=1
            )

            open_places = locations[open_mask]

            for place_type in place_types:
                rows.append({
                    "day_group": day_group,
                    "weekday_short": day,
                    "hour": hour,
                    "place_type": place_type,
                    "open_count": len(
                        open_places[open_places["place_type"] == place_type]
                    )
                })

open_by_type = pd.DataFrame(rows)

# ----------------------------
# 6. Connect opening hours with pedestrian movement
# ----------------------------

combined = open_by_type.merge(
    pedestrian_hourly,
    on=["day_group", "weekday_short", "hour"],
    how="left"
)

combined["avg_pedestrian_movement"] = combined["avg_pedestrian_movement"].fillna(0)

# This is the most important calculation:
# stronger value = more places open + more people present
combined["movement_while_open"] = (
    combined["open_count"] * combined["avg_pedestrian_movement"]
)

# Average values for each day group, place type, and hour
heatmap_data = (
    combined
    .groupby(["day_group", "place_type", "hour"], as_index=False)
    .agg(
        open_count=("open_count", "mean"),
        avg_pedestrian_movement=("avg_pedestrian_movement", "mean"),
        movement_while_open=("movement_while_open", "mean")
    )
)

# ----------------------------
# 7. Create Plotly heatmaps
# ----------------------------

output_folder = Path("plotly_heatmaps")
output_folder.mkdir(exist_ok=True)

for group in ["Weekday", "Weekend", "Public holiday"]:

    subset = heatmap_data[heatmap_data["day_group"] == group].copy()

    # Sort place types by strongest total activity
    order = (
        subset.groupby("place_type")["movement_while_open"]
        .sum()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    matrix = (
        subset
        .pivot(index="place_type", columns="hour", values="movement_while_open")
        .fillna(0)
        .reindex(order)
    )

    fig = px.imshow(
        matrix,
        labels={
            "x": "Hour of day",
            "y": "Type of place",
            "color": "Open places × pedestrian movement"
        },
        title=f"What was open when people were in the city? — {group}",
        aspect="auto",
        text_auto=".0f"
    )

    fig.update_layout(
        height=550,
        width=1000,
        title_x=0.5,
        xaxis=dict(
            tickmode="linear",
            tick0=0,
            dtick=1
        )
    )

    file_name = f"plotly_heatmap_{group.lower().replace(' ', '_')}.html"
    file_path = output_folder / file_name

    fig.write_html(file_path)

    print(f"Saved: {file_path}")

Saved: plotly_heatmaps\plotly_heatmap_weekday.html
Saved: plotly_heatmaps\plotly_heatmap_weekend.html
Saved: plotly_heatmaps\plotly_heatmap_public_holiday.html
